# Student fine-tuning: Qwen3-ASR Khmer LoRA
Train the published Khmer checkpoint with a **student-owned LoRA adapter** on exactly 531 manifested FLEURS training clips. Use exactly 124 validation clips for selection. The script does not load the held-out test split. A validation result is not a final test claim. CER and ICU Khmer WER must each be below 20% to exceed 80% correctness under these measures. The first 531-clip experiment has already been measured and did **not** improve the publisher checkpoint; see `results/diagnostic/qwen_lora_validation_v1.json`.


In [ ]:
!nvidia-smi
!pip install -q qwen-asr datasets torchcodec pyicu-wheels==2.15.2 jiwer soundfile accelerate peft
!pip uninstall -y torchao
from pathlib import Path
import subprocess, torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before training.')
repo = Path('/content/khmer_asr')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/Seypa-47/khmer_asr.git', str(repo)], check=True)
official = Path('/content/Qwen3-ASR')
if not official.exists():
    subprocess.run(['git', 'clone', 'https://github.com/QwenLM/Qwen3-ASR.git', str(official)], check=True)
subprocess.run(['git', '-C', str(official), 'checkout', '7c6daf77a2421100f5fb066495372c00129d39ff'], check=True)
%cd /content/khmer_asr


Mount Drive only after reviewing the code above. If it fails, the code uses runtime storage and prints a warning. Runtime storage disappears when Colab disconnects; download the adapter folder or reproduce the run locally before disconnecting.


In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
    output_root = Path('/content/drive/MyDrive/khmer_asr_final_runs')
except ValueError:
    output_root = Path('/content/khmer_asr_runtime_checkpoints')
    print('Drive mount failed. Download adapters before the runtime ends or reproduce locally.')
output_root.mkdir(parents=True, exist_ok=True)


In [ ]:
# First controlled epoch; checkpoints and validation predictions stay outside Git.
subprocess.run(['python', '-u', 'src/train_qwen_lora.py', '--output-dir', str(output_root / 'qwen_lora_v1'), '--epochs', '1', '--lr', '1e-4', '--grad-acc', '4'], check=True)


Compare the full validation CER/WER with the published checkpoint's 14.29% CER and 32.07% WER on the same 124 clips. Only after choosing an adapter should the separate held-out test be evaluated once. Do not present the publisher's pre-trained model as a student-trained approach.
